In [1]:
import pandas as pd

# Ruta al fichero CSV
to_path = r"D:\tfg_finance\mi_proyecto_datos\local_db\raw\AValue_anual_1997-2025_selected_union.csv"

# Lectura básica (asumiendo separador coma y codificación UTF-8)
#df = pd.read_csv(to_path)

# Si el separador es otro (por ejemplo ; )
df = pd.read_csv(to_path, sep=';', engine="python" )

print(df.columns.tolist())
print(df.head())


['date', 'date_q', 'av_total', 'av_third']
   date date_q av_total av_third
0  1997     Q1   8577,6   4743,9
1  1997     Q2   8577,6   4743,9
2  1997     Q3   8577,6   4743,9
3  1997     Q4   8577,6   4743,9
4  1998     Q1   9062,8   5099,1


In [3]:
import pandas as pd
from pathlib import Path

path = Path(r"D:\tfg_finance\mi_proyecto_datos\local_db\raw\AValue_anual_1997-2025_selected_union.csv")

df = pd.read_csv(path, sep=";", engine="python")
df.columns = df.columns.str.strip()

print(df.head())
print(df.dtypes)

   date date_q av_total av_third
0  1997     Q1   8577,6   4743,9
1  1997     Q2   8577,6   4743,9
2  1997     Q3   8577,6   4743,9
3  1997     Q4   8577,6   4743,9
4  1998     Q1   9062,8   5099,1
date        object
date_q      object
av_total    object
av_third    object
dtype: object


In [7]:
import pandas as pd
from pathlib import Path

path = Path(r"D:\tfg_finance\mi_proyecto_datos\local_db\raw\AValue_anual_1997-2025_selected_union.csv")

df = pd.read_csv(path, sep=";", engine="python")
df.columns = df.columns.str.strip()

print(df.head())
print(df["date"].unique()[:10])

   date date_q av_total av_third
0  1997     Q1   8577,6   4743,9
1  1997     Q2   8577,6   4743,9
2  1997     Q3   8577,6   4743,9
3  1997     Q4   8577,6   4743,9
4  1998     Q1   9062,8   5099,1
['1997' '1998' '1999' '2000' '2001' '2002' '2003' '2004' '2005' '2006']


In [8]:
# 1) Eliminar filas de cabecera repetida u otros textos
df = df[df["date"] != "date"].copy()

# 2) Convertir 'date' (año) a int
df["date"] = df["date"].astype(int)

# 3) Extraer trimestre de 'Q1', 'Q2', ...
df["quarter"] = df["date_q"].str.extract(r"Q(\d+)").astype(int)

# 4) Fecha fin de trimestre
df["date"] = pd.PeriodIndex.from_fields(
    year=df["date"].to_numpy(),
    quarter=df["quarter"].to_numpy(),
    freq="Q"
).to_timestamp("Q")

# 5) Opcional: quitar columnas auxiliares
df = df.drop(columns=["date_q", "quarter"])

In [9]:
for col in ["av_total", "av_third"]:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(".", "", regex=False)   # por si hay miles tipo 1.234,56
        .str.replace(",", ".", regex=False)  # coma -> punto
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")

df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 116 entries, 0 to 115
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   date      116 non-null    datetime64[ns]
 1   av_total  116 non-null    float64       
 2   av_third  116 non-null    float64       
dtypes: datetime64[ns](1), float64(2)
memory usage: 3.6 KB


,date,av_total,av_third
0,1997-03-31,8577.6,4743.9
1,1997-06-30,8577.6,4743.9
2,1997-09-30,8577.6,4743.9
3,1997-12-31,8577.6,4743.9
4,1998-03-31,9062.8,5099.1


In [10]:
from pathlib import Path

# Asegúrate de que df ya está limpio con date (datetime) y av_total, av_third en float

interim_dir = Path(r"D:\tfg_finance\mi_proyecto_datos\local_db\interim")
interim_dir.mkdir(parents=True, exist_ok=True)

csv_path = interim_dir / "AValue_union_quarterly_panel.csv"

df.to_csv(csv_path, index=False)
print(csv_path)

D:\tfg_finance\mi_proyecto_datos\local_db\interim\AValue_union_quarterly_panel.csv


In [11]:
import numpy as np

# Si todavía no lo hiciste, asegúrate de tener df cargado desde el CSV limpio/Parquet
# y av_total, av_third como float.

den = df["av_total"].replace(0, np.nan)  # evitar división por 0
df["av_ratio"] = df["av_third"] / den

In [12]:
df[["date", "av_total", "av_third", "av_ratio"]].head()

,date,av_total,av_third,av_ratio
0,1997-03-31,8577.6,4743.9,0.553057
1,1997-06-30,8577.6,4743.9,0.553057
2,1997-09-30,8577.6,4743.9,0.553057
3,1997-12-31,8577.6,4743.9,0.553057
4,1998-03-31,9062.8,5099.1,0.562641


In [13]:
from pathlib import Path

interim_dir = Path(r"D:\tfg_finance\mi_proyecto_datos\local_db\interim")
csv_path = interim_dir / "AValue_union_quarterly_panel.csv"

df.to_csv(csv_path, index=False)
print(csv_path)

D:\tfg_finance\mi_proyecto_datos\local_db\interim\AValue_union_quarterly_panel.csv
